In [1]:
%pip install scikit-learn==1.3.2 pandas==2.1.4 streamlit joblib numpy

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ========================================
#                 Import
# ========================================

import pandas as pd 
import numpy as np

from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report,confusion_matrix,roc_auc_score

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    precision_recall_curve,
    confusion_matrix,
    RocCurveDisplay
)

import shap
import matplotlib.pyplot as plt


c:\Users\vinit\OneDrive\Desktop\Customer-Churn-Prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("telco.csv")

#Drop customerID

df = df.drop("Customer ID",axis=1)

df.head()


,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents,Country,State,City,...,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Satisfaction Score,Customer Status,Churn Label,Churn Score,CLTV,Churn Category,Churn Reason
0,Male,78,No,Yes,No,No,0,United States,California,Los Angeles,...,20,0.00,59.65,3,Churned,Yes,91,5433,Competitor,Competitor offered more data
1,Female,74,No,Yes,Yes,Yes,1,United States,California,Los Angeles,...,0,390.80,1024.10,3,Churned,Yes,69,5302,Competitor,Competitor made better offer
2,Male,71,No,Yes,No,Yes,3,United States,California,Los Angeles,...,0,203.94,1910.88,2,Churned,Yes,81,3179,Competitor,Competitor made better offer
3,Female,78,No,Yes,Yes,Yes,1,United States,California,Inglewood,...,0,494.00,2995.07,2,Churned,Yes,88,5337,Dissatisfaction,Limited range of services
4,Female,80,No,Yes,Yes,Yes,1,United States,California,Whittier,...,0,234.21,3102.36,2,Churned,Yes,67,2793,Price,Extra data charges


In [4]:
# 1. Drop pure leakage columns
df = df.drop(columns=[
    "Customer Status",
    "Churn Score",
    "CLTV",
    "Churn Category",
    "Churn Reason"
])

# 2. Fix target properly
df["Churn Label"] = df["Churn Label"].astype(str).str.strip()

print(df["Churn Label"].unique())   # must show ['Yes','No']

df["Churn Label"] = df["Churn Label"].map({"Yes":1, "No":0})

print(df["Churn Label"].value_counts())

['Yes' 'No']
Churn Label
0    5174
1    1869
Name: count, dtype: int64


In [5]:
print(df.shape)
print(df["Churn Label"].unique())

(7043, 44)
[1 0]


In [6]:
df = df.drop(columns=[
    "Churn Score",
    "CLTV",
    "Customer Status",
    "Total Revenue",
    "Total Charges",
    "Total Refunds",
    "Total Extra Data Charges",
    "Total Long Distance Charges",
    "Tenure in Months",
    "Satisfaction Score"

], errors="ignore")

In [7]:
print(df.shape)
print(df["Churn Label"].value_counts())

(7043, 37)
Churn Label
0    5174
1    1869
Name: count, dtype: int64


In [8]:
df["Offer"] = df["Offer"].fillna("No Offer")

#Filling in that NO Internet
df["Internet Type"] = df["Internet Type"].fillna("No Internet")

#If numeric columns stll have nulls,inspect beafore droppin

print(df.isna().sum().sort_values(ascending=False).head())

Gender                               0
Avg Monthly Long Distance Charges    0
Internet Service                     0
Internet Type                        0
Avg Monthly GB Download              0
dtype: int64


In [23]:
#=================================
#3. Spilt features / target
#=================================

X=df.drop("Churn Label",axis=1)
y=df["Churn Label"]

X_train, X_test, y_train , y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [24]:
#=================================
#Preprocessing 
#=================================

categorical_cols =X.select_dtypes(include="object").columns
numeric_cols = X.select_dtypes(include=["int64","float64"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num",StandardScaler(),numeric_cols),
        ("cat",OneHotEncoder(handle_unknown="ignore"),categorical_cols)
    ]
)

In [ ]:
# #==================================
# #Logistic Regression 
# #==================================

# log_model =Pipeline(steps=[
#     ("preprocessor",preprocessor),
#     ("classifier",LogisticRegression(max_iter=1000,class_weight="balanced",solver="liblinear"))
# ])

# log_model.fit(X_train,y_train)

# y_pred = log_model.predict(X_test)
# y_prob = log_model.predict_proba(X_test)[:,1]

# print(y_train.value_counts())
# print(y_test.value_counts())

# # Metrics
# print("classification Report:\n")
# print(classification_report(y_test,y_pred))

# print("confusion_matrix:\n")
# print(confusion_matrix(y_test,y_pred))

# print("ROC=AUC:",roc_auc_score(y_test,y_prob))

Churn Label
0    4139
1    1495
Name: count, dtype: int64
Churn Label
0    1035
1     374
Name: count, dtype: int64
classification Report:

              precision    recall  f1-score   support

           0       0.93      0.80      0.86      1035
           1       0.60      0.83      0.70       374

    accuracy                           0.81      1409
   macro avg       0.77      0.82      0.78      1409
weighted avg       0.84      0.81      0.82      1409

confusion_matrix:

[[829 206]
 [ 62 312]]
ROC=AUC: 0.8940530625952621


In [30]:
#=================================
#Random Forest Pipline
#=================================

rf_model = Pipeline(steps=[
    ("preprocessor",preprocessor), #same preprocessing as LR
    ("classifier",RandomForestClassifier(
        n_estimators=200, #Number of trees
        max_depth=None,
        class_weight="balanced",#Handle churn imbalance
        random_state=42
    ))
])

#Train
rf_model.fit(X_train,y_train)

#Predict
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:,1]

#Metircs
print("Train/Test class distribution:")
print(y_train.value_counts())
print(y_test.value_counts())

print("\nClassifiction Report:\n")
print(classification_report(y_test,y_pred_rf))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test,y_pred_rf))

print("ROC-AUC:",roc_auc_score(y_test,y_prob_rf))

Train/Test class distribution:
Churn Label
0    4139
1    1495
Name: count, dtype: int64
Churn Label
0    1035
1     374
Name: count, dtype: int64

Classifiction Report:

              precision    recall  f1-score   support

           0       0.85      0.93      0.89      1035
           1       0.73      0.53      0.62       374

    accuracy                           0.82      1409
   macro avg       0.79      0.73      0.75      1409
weighted avg       0.82      0.82      0.82      1409

Confusion Matrix:

[[962  73]
 [174 200]]
ROC-AUC: 0.8859283370792322


In [27]:
import joblib
feature_dtypes = X.dtypes
joblib.dump(feature_dtypes,"feature_dtypes.pkl")

['feature_dtypes.pkl']

In [ ]:
import joblib
joblib.dump(rf_model,"churn_model.pkl")

['churn_model.pkl']